# Post-Hoc Calibration Map Ablation

Compares calibration methods (Platt-Uni, Platt-Bi, Isotonic, Hist-Bin) applied to signal-space confidence (LC, TP, SU) across all datasets and models. Train: first 30%, Test: last 70%. Reports generalised ECE and FD on the test split.

In [ ]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
sys.path.insert(0, '/home/ivan/lm-confidence-evaluation-harness')

warnings.filterwarnings('ignore')

/home/ivan/miniconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from lm_conf.default_utils.custom_types import OrganisedOutputs
from lm_conf.post_processing.metrics import generalised_ece, faithfulness_divergence
from calibration.utils import (
    in_domain_numerical_post_hoc_calibration,
    _print_signal_metrics_from_lists,
)

## Helper Functions

In [ ]:
COMMON_DIR = '/hdd/ivny'
DATASETS = ['mmlu', 'squadv2', 'truthful_qa']
SIGNALS = {'lc': 'LC', 'tp': 'TP', 'su': 'SU'}
METHODS = ['platt_uni', 'platt_bi', 'isotonic', 'hist_bin']
METHOD_LABELS = {
    'platt_uni': 'Platt-Uni',
    'platt_bi':  'Platt-Bi',
    'isotonic':  'Isotonic',
    'hist_bin':  'Hist-Bin',
}
DATASET_LABELS = {
    'mmlu': 'MMLU',
    'squadv2': 'SQuAD 2.0',
    'truthful_qa': 'TruthfulQA',
}


def _leaf_dirs(root_dir):
    """Return all leaf directories (no sub-dirs) under root_dir, sorted."""
    leaf_dirs = []
    for root, dirs, files in os.walk(root_dir):
        if not dirs:
            leaf_dirs.append(root)
    return sorted(leaf_dirs)


def _get_latest_leaf(parent_dir: str) -> str | None:
    """
    If parent_dir has multiple child dirs (siblings), return the one with the
    most recent mtime; otherwise return the single child. Returns None if
    graded_outputs_0.pkl is missing in the chosen dir.
    """
    try:
        subdirs = [
            os.path.join(parent_dir, d)
            for d in os.listdir(parent_dir)
            if os.path.isdir(os.path.join(parent_dir, d))
        ]
    except FileNotFoundError:
        return None
    if not subdirs:
        return None
    chosen = max(subdirs, key=os.path.getmtime)
    pkl = os.path.join(chosen, 'graded_outputs_0.pkl')
    return chosen if os.path.exists(pkl) else None


def load_pkl(path: str, filename: str):
    with open(os.path.join(path, filename), 'rb') as f:
        return pickle.load(f)


def compute_metrics(confidences, accuracies, answers):
    """
    Compute generalised ECE and mean FD from lists of BetaDistribution objects.
    Filters out None confidences.
    """
    valid = [
        (c, a, ans)
        for c, a, ans in zip(confidences, accuracies, answers)
        if c is not None
    ]
    if not valid:
        return float('nan'), float('nan'), 0
    confs, accs, anss = zip(*valid)
    org = OrganisedOutputs(
        accuracy_scores=[list(accs)],
        extracted_confidences=[list(confs)],
        extracted_answers=[list(anss)],
    )
    ece = generalised_ece({}, org)[0]
    fd  = faithfulness_divergence({}, org)[0]
    return float(ece), float(fd), len(valid)

## Discover Runs

Walk `/hdd/ivny/results/{dataset}/direct_qa_{signal}/{org}/{model}/` and select the latest timestamp dir per model.

In [ ]:
runs = []  # list of dicts: dataset, signal, model, leaf_dir

for dataset in DATASETS:
    results_root = os.path.join(COMMON_DIR, 'results', dataset)
    dqa_dirs = sorted([
        d for d in os.listdir(results_root)
        if d.startswith('direct_qa_')
    ])
    for dqa in dqa_dirs:
        # Extract signal suffix e.g. 'direct_qa_unified_lc' -> 'lc'
        suffix = dqa.replace('direct_qa_unified_', '').replace('direct_qa_', '')
        if suffix not in SIGNALS:
            continue
        dqa_path = os.path.join(results_root, dqa)
        # Walk org/model levels
        for org in os.listdir(dqa_path):
            org_path = os.path.join(dqa_path, org)
            if not os.path.isdir(org_path):
                continue
            for model in os.listdir(org_path):
                model_path = os.path.join(org_path, model)
                if not os.path.isdir(model_path):
                    continue
                leaf = _get_latest_leaf(model_path)
                if leaf is None:
                    continue
                runs.append({
                    'dataset': dataset,
                    'signal':  suffix,
                    'model':   f'{org}/{model}',
                    'leaf':    leaf,
                })

print(f'Total runs found: {len(runs)}')
pd.DataFrame(runs)[['dataset', 'signal', 'model']].value_counts().reset_index()

Total runs found: 63


,dataset,signal,model,count
0,mmlu,lc,mistralai/Mistral-7B-Instruct-v0.3,1
1,mmlu,lc,google/gemma-4-31B-it,1
2,mmlu,lc,meta-llama/Llama-3.1-8B-Instruct,1
3,mmlu,lc,openai/gpt-oss-20b,1
4,mmlu,lc,openai/gpt-oss-120b,1
...,...,...,...,...
58,truthful_qa,tp,meta-llama/Llama-3.1-8B-Instruct,1
59,truthful_qa,tp,openai/gpt-oss-20b,1
60,truthful_qa,tp,openai/gpt-oss-120b,1
61,truthful_qa,tp,qwen/Qwen3-235B-A22B-Instruct-2507-tput,1


## Run Calibration Ablation

For each run, load `graded_outputs_0.pkl`, apply each calibration method (first 30% train, last 70% test), and record ECE and FD on the test portion.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _valid_acc(a):
    if a is None or a == '':
        return False
    try:
        v = float(a)
        return not np.isnan(v) and v in (0.0, 1.0)
    except (ValueError, TypeError):
        return False


def process_run(run):
    try:
        graded: OrganisedOutputs = load_pkl(run['leaf'], 'graded_outputs_0.pkl')
    except Exception:
        return []

    raw_confs = graded.extracted_confidences[0]
    raw_accs  = graded.accuracy_scores[0]
    raw_ans   = graded.extracted_answers[0]

    valid_mask = [c is not None and _valid_acc(a) for c, a in zip(raw_confs, raw_accs)]
    confs = [c for c, v in zip(raw_confs, valid_mask) if v]
    accs  = [float(a) for a, v in zip(raw_accs, valid_mask) if v]
    ans   = [a for a, v in zip(raw_ans, valid_mask) if v]
    if len(confs) < 10:
        return []

    n_train = max(1, int(np.ceil(0.30 * len(confs))))
    rows = []

    pre_ece, pre_fd, n_test = compute_metrics(confs[n_train:], accs[n_train:], ans[n_train:])
    rows.append({'dataset': run['dataset'], 'signal': run['signal'], 'model': run['model'],
                 'method': 'uncalibrated', 'ECE': pre_ece, 'FD': pre_fd, 'n_test': n_test})

    confs_arr = np.array(confs, dtype=object)
    accs_arr  = np.array(accs, dtype=float)

    for method in METHODS:
        try:
            cal_confs = in_domain_numerical_post_hoc_calibration(confs_arr, accs_arr, method=method)
            post_ece, post_fd, n = compute_metrics(cal_confs[n_train:], accs[n_train:], ans[n_train:])
        except Exception:
            post_ece, post_fd, n = float('nan'), float('nan'), 0
        rows.append({'dataset': run['dataset'], 'signal': run['signal'], 'model': run['model'],
                     'method': method, 'ECE': post_ece, 'FD': post_fd, 'n_test': n})
    return rows


records = []
with ThreadPoolExecutor(max_workers=None) as pool:
    futures = {pool.submit(process_run, run): run for run in runs}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Calibrating'):
        records.extend(fut.result())

df_records = pd.DataFrame(records)
print(f'Total records: {len(df_records)}')
df_records.head()

Calibrating:   0%|          | 0/63 [00:00<?, ?it/s]

[platt_uni_train_calibrated] n=4213 | ECE=0.124047 | FD=0.448709
[platt_uni_train_calibrated] n=4213 | ECE=0.064381 | FD=0.485607
[platt_uni_train_calibrated] n=4154 | ECE=0.082234 | FD=0.376395
[platt_uni_train_calibrated] n=4212 | ECE=0.083125 | FD=0.394378
[platt_uni_train_calibrated] n=4213 | ECE=0.074815 | FD=0.417430
[platt_uni_train_calibrated] n=4213 | ECE=0.118730 | FD=0.427142
[platt_uni_train_calibrated] n=4207 | ECE=0.087436 | FD=0.415373
[platt_uni_train_calibrated] n=4213 | ECE=0.120645 | FD=0.457481
[platt_bi_train_calibrated] n=4213 | ECE=0.123184 | FD=0.449954
[platt_bi_train_calibrated] n=4154 | ECE=0.081170 | FD=0.371418
[platt_bi_train_calibrated] n=4213 | ECE=0.064161 | FD=0.484913
[platt_bi_train_calibrated] n=4212 | ECE=0.083344 | FD=0.400986
[platt_bi_train_calibrated] n=4213 | ECE=0.074766 | FD=0.417093
[platt_bi_train_calibrated] n=4213 | ECE=0.118319 | FD=0.429257
[platt_bi_train_calibrated] n=4207 | ECE=0.086304 | FD=0.405756
[platt_bi_train_calibrated] n=42

Calibrating:  13%|█▎        | 8/63 [06:40<10:24, 11.36s/it]   

[platt_uni_train_calibrated] n=4213 | ECE=0.057997 | FD=0.472409
[platt_uni_train_calibrated] n=4212 | ECE=0.042253 | FD=0.445263
[platt_uni_train_calibrated] n=4213 | ECE=0.034790 | FD=0.476104
[platt_uni_train_calibrated] n=4207 | ECE=0.025741 | FD=0.407213
[platt_uni_train_calibrated] n=4154 | ECE=0.023877 | FD=0.405587
[platt_uni_train_calibrated] n=4213 | ECE=0.041924 | FD=0.436496
[platt_uni_train_calibrated] n=4213 | ECE=0.092705 | FD=0.456535
[platt_uni_train_calibrated] n=4212 | ECE=0.008413 | FD=0.436073
[platt_bi_train_calibrated] n=4213 | ECE=0.057653 | FD=0.472230
[platt_bi_train_calibrated] n=4212 | ECE=0.041931 | FD=0.444817
[platt_bi_train_calibrated] n=4213 | ECE=0.034830 | FD=0.472666
[platt_bi_train_calibrated] n=4154 | ECE=0.023523 | FD=0.405053
[platt_bi_train_calibrated] n=4207 | ECE=0.025276 | FD=0.404990
[platt_bi_train_calibrated] n=4213 | ECE=0.092639 | FD=0.457932
[platt_bi_train_calibrated] n=4213 | ECE=0.041826 | FD=0.436489
[platt_bi_train_calibrated] n=42

Calibrating:  13%|█▎        | 8/63 [10:12<1:10:13, 76.60s/it]


## Aggregate Results

Mean ECE and FD across all (dataset, model) combinations, grouped by (signal, method).

In [ ]:
METHOD_ORDER = ['uncalibrated'] + METHODS
SIGNAL_ORDER = ['lc', 'tp', 'su']

# Aggregate: mean ECE and FD over all (dataset, model) per (signal, method)
agg = (
    df_records
    .groupby(['signal', 'method'], sort=False)[['ECE', 'FD']]
    .agg(['mean', 'std'])
    .round(4)
)
agg.columns = ['ECE_mean', 'ECE_std', 'FD_mean', 'FD_std']
agg = agg.reset_index()

# Pivot: rows = method, columns = (signal, metric)
pivot_rows = []
for method in METHOD_ORDER:
    row = {'Method': METHOD_LABELS.get(method, 'Uncalibrated')}
    for sig in SIGNAL_ORDER:
        sub = agg[(agg['signal'] == sig) & (agg['method'] == method)]
        if sub.empty:
            row[f'{SIGNALS[sig]}_ECE'] = '-'
            row[f'{SIGNALS[sig]}_FD']  = '-'
        else:
            ece_m = sub['ECE_mean'].values[0]
            ece_s = sub['ECE_std'].values[0]
            fd_m  = sub['FD_mean'].values[0]
            fd_s  = sub['FD_std'].values[0]
            row[f'{SIGNALS[sig]}_ECE'] = f'{ece_m:.4f} ± {ece_s:.4f}'
            row[f'{SIGNALS[sig]}_FD']  = f'{fd_m:.4f} ± {fd_s:.4f}'
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index('Method')
pivot_df.columns = pd.MultiIndex.from_tuples(
    [(sig, metric) for sig in ['LC', 'TP', 'SU'] for metric in ['ECE', 'FD']]
)
print('Aggregated results (mean ± std across all datasets and models):')
pivot_df

## Per-Signal Summary Table (means only)

In [ ]:
# Clean numeric table for visual inspection
summary_rows = []
for method in METHOD_ORDER:
    row = {'Method': METHOD_LABELS.get(method, 'Uncalibrated')}
    for sig in SIGNAL_ORDER:
        sub = agg[(agg['signal'] == sig) & (agg['method'] == method)]
        if sub.empty:
            row[f'{SIGNALS[sig]} ECE'] = float('nan')
            row[f'{SIGNALS[sig]} FD']  = float('nan')
        else:
            row[f'{SIGNALS[sig]} ECE'] = round(float(sub['ECE_mean'].values[0]), 4)
            row[f'{SIGNALS[sig]} FD']  = round(float(sub['FD_mean'].values[0]), 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Method')

def highlight_best(col):
    """Highlight the row with the lowest value in each column."""
    is_best = col == col.min()
    return ['font-weight: bold; color: #1a7a1a' if v else '' for v in is_best]

display(summary_df.style.apply(highlight_best, axis=0).format('{:.4f}'))

## Per-Dataset Breakdown

In [ ]:
for dataset in DATASETS:
    ds_df = df_records[df_records['dataset'] == dataset]
    ds_agg = (
        ds_df.groupby(['signal', 'method'])[['ECE', 'FD']]
        .mean().round(4).reset_index()
    )

    rows = []
    for method in METHOD_ORDER:
        row = {'Method': METHOD_LABELS.get(method, 'Uncalibrated')}
        for sig in SIGNAL_ORDER:
            sub = ds_agg[(ds_agg['signal'] == sig) & (ds_agg['method'] == method)]
            if sub.empty:
                row[f'{SIGNALS[sig]} ECE'] = float('nan')
                row[f'{SIGNALS[sig]} FD']  = float('nan')
            else:
                row[f'{SIGNALS[sig]} ECE'] = round(float(sub['ECE'].values[0]), 4)
                row[f'{SIGNALS[sig]} FD']  = round(float(sub['FD'].values[0]), 4)
        rows.append(row)

    tbl = pd.DataFrame(rows).set_index('Method')
    print(f'\n=== {DATASET_LABELS[dataset]} ===')
    display(tbl.style.apply(highlight_best, axis=0).format('{:.4f}'))

## Delta Table (post − pre calibration)

In [ ]:
# Compute delta: calibrated method minus uncalibrated, for each (dataset, model, signal)
df_uncal = df_records[df_records['method'] == 'uncalibrated'].set_index(
    ['dataset', 'signal', 'model']
)[['ECE', 'FD']].rename(columns={'ECE': 'ECE_base', 'FD': 'FD_base'})

df_cal = df_records[df_records['method'] != 'uncalibrated'].copy()
df_cal = df_cal.join(df_uncal, on=['dataset', 'signal', 'model'])
df_cal['ΔECE'] = df_cal['ECE'] - df_cal['ECE_base']
df_cal['ΔFD']  = df_cal['FD']  - df_cal['FD_base']

delta_agg = (
    df_cal.groupby(['signal', 'method'])[['ΔECE', 'ΔFD']]
    .mean().round(4).reset_index()
)

delta_rows = []
for method in METHODS:
    row = {'Method': METHOD_LABELS[method]}
    for sig in SIGNAL_ORDER:
        sub = delta_agg[(delta_agg['signal'] == sig) & (delta_agg['method'] == method)]
        if sub.empty:
            row[f'{SIGNALS[sig]} ΔECE'] = float('nan')
            row[f'{SIGNALS[sig]} ΔFD']  = float('nan')
        else:
            row[f'{SIGNALS[sig]} ΔECE'] = round(float(sub['ΔECE'].values[0]), 4)
            row[f'{SIGNALS[sig]} ΔFD']  = round(float(sub['ΔFD'].values[0]),  4)
    delta_rows.append(row)

delta_df = pd.DataFrame(delta_rows).set_index('Method')
print('Δ = calibrated − uncalibrated (negative = improvement)')

def highlight_delta(col):
    """Green for the most negative (biggest improvement), red for most positive."""
    styles = []
    for v in col:
        if v == col.min():
            styles.append('font-weight: bold; color: #1a7a1a')
        elif v == col.max():
            styles.append('color: #a00')
        else:
            styles.append('')
    return styles

display(delta_df.style.apply(highlight_delta, axis=0).format('{:+.4f}'))